# CANDIDATE FEATURE IDENTIFICATION
The purpose of this notebook is to aggregate in one location the potential features to include in the training set. 
Inputs: External Dataset Parquet Files. 
Desired Outputs: 
- Of redundant variables, select the most reliable and informative one. 
- Understand how to calculate any computed features.

Notes
- Use a separate .py module under src/external_dataset_name/ingest.py to pull in the data with API or other method and write to data/interim/external_dataset_name.parquet. 
- This notebook should read in just a small sample of each dataset for feature exploration. Reading in the full dataset may crash the Jupyter kernel. 

For each feature, attempt to identify:
- time of observation
- unit of observation / what it actually is
- geographic scope (i.e. is it at the census block group, tract, or zip code level?)
- missingness
- what preprocessing is required (e.g. geocoding to census block group, converting a raw count to density)

In [1]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
pd.set_option('display.max_columns', None)

## Lottery and Sales Data (for reference when merging)

In [10]:
lottery_locations_file_path = Path(".").resolve().parent / "data/final/location_data.parquet"

#locations_pq = pq.ParquetFile(lottery_locations_file_path)
#locations_first_100_rows = next(locations_pq.iter_batches(batch_size=100))
#locations_df = pa.Table.from_batches([locations_first_100_rows]).to_pandas()

locations_df = pd.read_parquet(str(lottery_locations_file_path))

In [11]:
locations_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid
15869,115958,OLD COUNTRY WINE & LIQUOR DEPO,207 GLEN COVE RD STE A,CARLE PLACE,11514,0,LIQUOR,-73.61667,40.74743,b'\x01\x01\x00\x00\x00(\x9br\x85wgR\xc0O\x06G\...,360593038004,36059303800
13158,111584,WALMART SUPERCENTER #2844,1549 RTE 9,CLIFTON PARK,12065,313,DEPARTMENT,-73.74896,42.84694,b'\x01\x01\x00\x00\x00\xc7\x80\xec\xf5\xeeoR\x...,360910624091,36091062409
7063,73686,APPLEBEE'S #31,2120 CHILI AVE,ROCHESTER,14624,291,RESTAURANT-HOTEL,-77.68920,43.13131,b'\x01\x01\x00\x00\x00\xce\x19Q\xda\x1blS\xc0\...,360550144002,36055014400
2121,7875,YASMINA DELI INC,19 FORT PL,STATEN ISLAND,10301,0,DELI-GROCERY,-74.07897,40.64197,b'\x01\x01\x00\x00\x00^\xd7/\xd8\r\x85R\xc0\xa...,360850011001,36085001100
6079,63974,7-ELEVEN #11211H,722 TOWNLINE ROAD,HAUPPAUGE,11788,30,CONV. STORE,-73.18490,40.82629,b'\x01\x01\x00\x00\x00\xf2A\xcff\xd5KR\xc0\x84...,361031458033,36103145803


In [12]:
lottery_sales_file_path = Path(".").resolve().parent / "data/final/sales_data.parquet"

#sales_pq = pq.ParquetFile(lottery_sales_file_path)
#sales_first_100_rows = next(sales_pq.iter_batches(batch_size=1000))
#sales_df = pa.Table.from_batches([sales_first_100_rows]).to_pandas()

sales_df = pd.read_parquet(str(lottery_sales_file_path))

In [13]:
sales_df.sample(5)

,agtno,total_sales,first_date,last_date,first_pos_date,last_pos_date,pos_sales_days,num_cal_days,num_active_days,days_before_first_sale,days_after_last_sale,avg_cal_daily_sales,avg_active_daily_sales,use_active_period,avg_daily_sales
6036,114629,16785.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,235.0,365,365.0,0.0,0.0,45.986301,45.986301,False,45.986301
6634,115334,1320195.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,3616.973973,3616.973973,False,3616.973973
4157,111724,244757.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,364.0,365,365.0,0.0,0.0,670.568493,670.568493,False,670.568493
7096,115828,236797.0,2025-05-19,2025-12-31,2025-05-19,2025-12-31,227.0,227,227.0,0.0,0.0,1043.158590,1043.158590,False,1043.158590
284,101214,1441717.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,3949.909589,3949.909589,False,3949.909589


## Overture Maps Data

In [ ]:
overture_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_overture.parquet"

overture_pq = pq.ParquetFile(overture_file_path)
overture_first_100_rows = next(overture_pq.iter_batches(batch_size=100))
overture_df = pa.Table.from_batches([overture_first_100_rows]).to_pandas()

In [15]:
overture_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,gas_station_within_0.5_mi,gas_station_within_1_mi,arts_and_entertainment_within_0.5_mi,arts_and_entertainment_within_1_mi,shopping_within_0.5_mi,shopping_within_1_mi,social_or_community_service_within_0.5_mi,social_or_community_service_within_1_mi,restaurant_within_0.5_mi,restaurant_within_1_mi,bar_within_0.5_mi,bar_within_1_mi
92,76475,QUICKLEES #12,349 W MORRIS ST,BATH,14810,-77.33299,42.33839,b'\x01\x01\x00\x00\x00H\xf9I\xb5OUS\xc0\x8f\xa...,361019611001,36101961100,2,2,0,0,0,0,1,9,0,1,0,2
88,102050,FELIO FOOD CORP,1161 SUTTER AVE,BROOKLYN,11208,-73.87513,40.67232,b'\x01\x01\x00\x00\x00\xe7oB!\x02xR\xc0.9\xee\...,360471194003,36047119400,3,10,1,5,4,8,4,14,2,10,0,3
74,111311,ADK FOOD & FUEL,1 COUNTRY MILE,FRANKFORT,13340,-75.08829,43.03850,b'\x01\x01\x00\x00\x00\x16\xa4\x19\x8b\xa6\xc5...,360430102022,36043010202,0,2,1,1,0,0,0,0,0,0,0,1
50,88721,A TO Z GROCERY & DELI INC,1450 CROSBY AVE,BRONX,10461,-73.83000,40.84207,b'\x01\x01\x00\x00\x00\x85\xebQ\xb8\x1euR\xc0\...,360050264001,36005026400,4,10,0,1,1,4,7,18,1,6,3,6
96,95155,ARIAS LIQUOR INC,1648 BRUCKNER BLVD,BRONX,10473,-73.87382,40.82335,b'\x01\x01\x00\x00\x00\xce\xa5\xb8\xaa\xecwR\x...,360050028001,36005002800,4,16,1,4,2,8,3,17,5,13,1,1


## LEHD Data

In [ ]:
rac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_rac.parquet"

rac_pq = pq.ParquetFile(rac_file_path)
rac_first_100_rows = next(rac_pq.iter_batches(batch_size=100))
rac_df = pa.Table.from_batches([rac_first_100_rows]).to_pandas()

In [17]:
rac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,rac_c000,rac_ce01,rac_ce02,rac_ce03
40,68056,7-ELEVEN #16361E,33 MIDDLE COUNTRY RD RTE 25,CORAM,11727,-72.97863,40.87459,b'\x01\x01\x00\x00\x00\x9b8\xb9\xdf\xa1>R\xc0N...,361031583091,36103158309,917.0,119.0,247.0,551.0
88,48170,DELHI DINER LLC,95 MAIN ST,DELHI,13753,-74.91717,42.27689,b'\x01\x01\x00\x00\x00\xd4\xb7\xcc\xe9\xb2\xba...,360259708004,36025970800,341.0,66.0,80.0,195.0


In [ ]:
wac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_wac.parquet"

wac_pq = pq.ParquetFile(wac_file_path)
wac_first_100_rows = next(wac_pq.iter_batches(batch_size=100))
wac_df = pa.Table.from_batches([wac_first_100_rows]).to_pandas()

In [19]:
wac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,wac_c000,wac_cns07,wac_cns18,wac_ce01,wac_ce02,wac_ce03
64,81101,SOFT TOUCH NEWSTAND,1 METROTECH CENTER,BROOKLYN,11201,-73.98723,40.69376,b'\x01\x01\x00\x00\x00S\xe8\xbc\xc6.\x7fR\xc0\...,360470011001,36047001100,21611.0,171.0,872.0,2036.0,2591.0,16984.0
94,76475,QUICKLEES #12,349 W MORRIS ST,BATH,14810,-77.33299,42.33839,b'\x01\x01\x00\x00\x00H\xf9I\xb5OUS\xc0\x8f\xa...,361019611001,36101961100,1017.0,265.0,169.0,202.0,322.0,493.0


## AADT Data

In [ ]:
aadt_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/data_aadt_2025.parquet"

aadt_pq = pq.ParquetFile(aadt_file_path)
aadt_first_100_rows = next(aadt_pq.iter_batches(batch_size=100))
aadt_df = pa.Table.from_batches([aadt_first_100_rows]).to_pandas()

In [21]:
aadt_df.sample(50)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,bg_geoid,tract_geoid,index_right,aadt_count,aadt_road_name,aadt_functional_class,aadt_lanes,dist_to_read_m
47,13705,MIS HIJOS DELI & GROCERY,2401 2ND AVE,NEW YORK,10035,-73.93444,40.80155,360610194004,36061019400,54008.0,57017.0,WILLIS AVE BRG,14,4:0||||,273.061072
68,11527,PAY-O-MATIC CORP #315,2351 2ND AVE,NEW YORK,10035,-73.93565,40.79991,360610194003,36061019400,54008.0,57017.0,WILLIS AVE BRG,14,4:0||||,426.049053
94,113105,NAZ DELI CORP,234 E 106TH ST,NEW YORK,10029,-73.94326,40.79098,360610170001,36061017000,54008.0,57017.0,WILLIS AVE BRG,14,4:0||||,1568.999398
83,113999,TRIPPLE STAR LIQUOR & WINE INC,2025 3RD AVE,NEW YORK,10029,-73.94228,40.79486,360610172001,36061017200,54008.0,57017.0,WILLIS AVE BRG,14,4:0||||,1206.716356
59,116190,BARRIO BODEGA 112 CORP,204 E 112TH ST,NEW YORK,10029,-73.94132,40.79516,360610180002,36061018000,54008.0,57017.0,WILLIS AVE BRG,14,4:0||||,1123.209291
77,107671,7-ELEVEN #37116A,2256 2ND AVE,NEW YORK,10029,-73.93789,40.79683,360610188002,36061018800,54008.0,57017.0,WILLIS AVE BRG,14,4:0||||,784.179347
44,116007,ANABELLE DELI GROCERY CORP,2121 2ND AVE,NEW YORK,10029,-73.94095,40.79265,360610180001,36061018000,54008.0,57017.0,WILLIS AVE BRG,14,4:0||||,1301.729400
23,113322,JAYS ONE STOP INC,283-285 FRONT ST,BINGHAMTON,13905,-75.91442,42.10851,360070003002,36007000300,55222.0,60857.0,NaN,12,3:2||||,319.075659
18,113876,APPLE FOOD & CONVENIENCE CORP,266 CHENANGO ST,BINGHAMTON,13901,-75.90524,42.10649,360070005003,36007000500,55222.0,60857.0,NaN,12,3:2||||,794.391799
46,113139,H&A WINE & SPIRITS,2336 2ND AVE,NEW YORK,10035,-73.93600,40.79942,360610194002,36061019400,54008.0,57017.0,WILLIS AVE BRG,14,4:0||||,479.171118


## EPA Data

In [ ]:
epa_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_epa.parquet"

epa_pq = pq.ParquetFile(epa_file_path)
epa_first_100_rows = next(epa_pq.iter_batches(batch_size=100))
epa_df = pa.Table.from_batches([epa_first_100_rows]).to_pandas()

In [21]:
epa_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,OBJECTID,GEOID10,GEOID20,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,CBSA_POP,CBSA_EMP,HH,D3B,NatWalkInd,STATEFP2,COUNTYFP2,TRACTCE2,BLKGRPCE2
5,77603,SUPER D,357 N MAIN ST,FREEPORT,11520,-73.58764,40.66898,b'\x01\x01\x00\x00\x00\x92t\xcd\xe4\x9beR\xc0\...,360594141005,36059414100,157336,3.605940e+11,3.605940e+11,36,59,414100,5,19318471,9159786,327.0,185.123984,15.500000,36,059,414100,5
93,77429,RUNWAY #6371,6371 ROUTE 167,DOLGEVILLE,13329,-74.78544,43.09375,"b'\x01\x01\x00\x00\x00\x18>""\xa6D\xb2R\xc0\x00...",360430110023,36043011002,157376,3.604300e+11,3.604300e+11,36,43,11002,3,293287,122479,361.0,0.922333,5.333333,36,043,011002,3
55,37654,APPLE FOOD & GROCERY INC,340 MAIN ST,MIDDLEBURGH,12122,-74.33595,42.59917,b'\x01\x01\x00\x00\x00\xd6\xc5m4\x80\x95R\xc0F...,360957407004,36095740700,149708,3.609570e+11,3.609570e+11,36,95,740700,4,880481,447103,557.0,1.565185,5.000000,36,095,740700,4
13,100075,965 A & A DELI GROCERY CORP,965 MORRIS PARK AVE,BRONX,10462,-73.85730,40.84763,b'\x01\x01\x00\x00\x00\x17\xb7\xd1\x00\xdevR\x...,360050244002,36005024400,161219,3.600500e+11,3.600500e+11,36,5,24400,2,19318471,9159786,431.0,98.548492,14.500000,36,005,024400,2
35,111759,TT2 TRADING INC,376 S BAYVIEW AVE,FREEPORT,11520,-73.59283,40.64398,b'\x01\x01\x00\x00\x00\x9d\x85=\xed\xf0eR\xc0\...,360594145012,36059414501,157438,3.605940e+11,3.605940e+11,36,59,414501,2,19318471,9159786,487.0,183.901022,19.000000,36,059,414501,2
